# 08 - Precision-Recall Curves

In this notebook, we explore why Precision-Recall (PR) Curves are superior to ROC curves when dealing with highly imbalanced datasets (like Fraud Detection).

## Concept Overview
ROC curves use True Negatives in their math. If you have 1 million normal transactions and 100 frauds, the massive number of True Negatives dilutes the False Positives. PR Curves completely ignore True Negatives.

## Visual Demonstration: ROC vs PR on Imbalanced Data
Let's generate a highly imbalanced dataset and train a mediocre model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc, average_precision_score
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Generate Imbalanced Data (1% positive class)
X, y = make_classification(n_samples=10000, weights=[0.99, 0.01], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Model
model = LogisticRegression()
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

Now, let's plot BOTH the ROC curve and the PR curve for this exact same model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color='blue', label=f'ROC AUC = {roc_auc:.2f}')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('ROC Curve (Looks Amazing!)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# 2. PR Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
baseline = sum(y_test) / len(y_test) # Baseline is the positive class ratio

axes[1].plot(recall, precision, color='red', label=f'PR AUC (AP) = {pr_auc:.2f}')
axes[1].axhline(y=baseline, color='k', linestyle='--', label=f'Baseline = {baseline:.2f}')
axes[1].set_title('PR Curve (Looks Terrible!)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.show()

## Interpretation
The ROC Curve is lying to us. It says the model is fantastic (AUC 0.89). 
But the PR Curve exposes the truth. The Average Precision (AUC-PR) is only 0.28! Because the data is so imbalanced, the model is generating tons of False Positives, which destroys Precision but barely affects the False Positive Rate on the ROC curve.

## Scikit-Learn Implementation
Notice that we use `average_precision_score` in Scikit-Learn to calculate the area under the PR curve. We do not use the `auc()` function, as it uses linear interpolation which is overly optimistic for PR curves.

## Industry Notes
If your problem involves looking for needles in a haystack (fraud, anomalies, rare diseases), **NEVER present an ROC curve to stakeholders**. You must present a PR curve. The goal is to push the PR curve to the top-right corner.

## Exercises
1. Try balancing the dataset by passing `class_weight='balanced'` to the LogisticRegression model. Re-run the PR curve. Does it improve?
2. Change the dataset weights from `[0.99, 0.01]` to `[0.5, 0.5]` (perfectly balanced). How do the ROC and PR curves compare now?